# IDS 706 — Week 12 Mini Assignment: PySpark Data Processing

**Project:** Historical US Airline Flight Delay Analysis  
**Course:** IDS 706 — Data Engineering Systems  
**Date:** November 23, 2026

## Project Overview

This project uses PySpark in Databricks to analyze historical US airline flight records from 1999–2008. The dataset contains approximately 64.8 million records and exceeds 6 GB in source-file size, making it suitable for distributed data processing.

The pipeline demonstrates data loading, filtering, column transformations, aggregations, Spark SQL, query optimization, lazy evaluation, and partitioned Parquet output. An optional Spark MLlib classification model is also included to predict flight delays.

In [0]:

from pyspark.sql import functions as F
import time

# Test Spark
spark.range(3).show()


+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



In [0]:

# Check access to the original class dataset
flights_path = "/databricks-datasets/asa/airlines/"

display(dbutils.fs.ls(flights_path))


path,name,size,modificationTime
dbfs:/databricks-datasets/asa/airlines/1987.csv,1987.csv,127162942,1596560063000
dbfs:/databricks-datasets/asa/airlines/1988.csv,1988.csv,501039472,1596560063000
dbfs:/databricks-datasets/asa/airlines/1989.csv,1989.csv,486518821,1596560063000
dbfs:/databricks-datasets/asa/airlines/1990.csv,1990.csv,509194687,1596560063000
dbfs:/databricks-datasets/asa/airlines/1991.csv,1991.csv,491210093,1596560073000
dbfs:/databricks-datasets/asa/airlines/1992.csv,1992.csv,492313731,1596560082000
dbfs:/databricks-datasets/asa/airlines/1993.csv,1993.csv,490753652,1596560100000
dbfs:/databricks-datasets/asa/airlines/1994.csv,1994.csv,501558665,1596560101000
dbfs:/databricks-datasets/asa/airlines/1995.csv,1995.csv,530751568,1596560102000
dbfs:/databricks-datasets/asa/airlines/1996.csv,1996.csv,533922363,1596560113000


## 1. Dataset Loading and Exploration

**Dataset:** Historical US airline on-time performance records (1999–2008)  
**Source:** Databricks sample datasets, `/databricks-datasets/asa/airlines/`

The source contains 10 annual CSV files, approximately 6.29 GB in total, with 64,813,341 records and 29 columns.

The data includes flight dates, airline identifiers, departure and arrival airports, flight distances, delay measurements, and cancellation/diversion indicators.

PySpark loads the CSV files into a distributed DataFrame. Schema inference identifies the column types, and an initial inspection verifies the dataset's size and structure.

In [0]:

# Load flight records from 1999–2008
flights = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/databricks-datasets/asa/airlines/{1999,200*}.csv")
)

print("Number of rows:", flights.count())
print("Number of columns:", len(flights.columns))

flights.printSchema()
display(flights.limit(5))


Number of rows: 64813341
Number of columns: 29
root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- ArrTime: string (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- ActualElapsedTime: string (nullable = true)
 |-- CRSElapsedTime: string (nullable = true)
 |-- AirTime: string (nullable = true)
 |-- ArrDelay: string (nullable = true)
 |-- DepDelay: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- TaxiIn: string (nullable = true)
 |-- TaxiOut: string (nullable = true)
 |-- Cancelled: integer (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |

Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
1999,1,27,3,1906,1908,2024,2005,US,1244,N942VJ,78,57,66,19,-2,RIC,PHL,198,3,9,0,NA,0,NA,NA,NA,NA,NA
1999,1,28,4,2016,1908,2126,2005,US,1244,N955VJ,70,57,41,81,68,RIC,PHL,198,19,10,0,NA,0,NA,NA,NA,NA,NA
1999,1,29,5,1907,1908,2000,2005,US,1244,N929VJ,53,57,43,-5,-1,RIC,PHL,198,2,8,0,NA,0,NA,NA,NA,NA,NA
1999,1,31,7,1932,1908,2031,2005,US,1244,N912VJ,59,57,45,26,24,RIC,PHL,198,6,8,0,NA,0,NA,NA,NA,NA,NA
1999,1,1,5,1601,1535,1707,1645,US,297,N935VJ,66,70,47,22,26,ROC,PHL,257,11,8,0,NA,0,NA,NA,NA,NA,NA


## 2. Data Cleaning and Transformations

The pipeline prepares the flight records using several PySpark DataFrame transformations:

- Filters out cancelled and diverted flights.
- Selects only columns needed for the analysis.
- Converts arrival and departure delays to numeric values using `try_cast`.
- Removes records with missing or invalid delay measurements.
- Creates a binary `IsDelayed` column using `withColumn`, where 1 indicates an arrival delay greater than 15 minutes.

Filtering and column selection are performed early to reduce the amount of data processed by subsequent operations.

These operations are lazy transformations: Spark builds an execution plan but does not immediately execute the computation.

In [0]:

# Select relevant columns, filter records, and create new columns
flights_clean = (
    flights
    # Filter 1: Remove cancelled flights
    .filter(F.col("Cancelled") == 0)

    # Filter 2: Remove diverted flights
    .filter(F.col("Diverted") == 0)

    # Select only columns needed for analysis
    .select(
        "Year", "Month", "UniqueCarrier",
        "Origin", "Dest", "Distance",
        "ArrDelay", "DepDelay"
    )

    # Convert delay columns to numeric types
    .withColumn("ArrDelay", F.expr("try_cast(ArrDelay as double)"))
    .withColumn("DepDelay", F.expr("try_cast(DepDelay as double)"))

    # Filter 3: Remove records with missing delay values
    .filter(
        F.col("ArrDelay").isNotNull() &
        F.col("DepDelay").isNotNull()
    )

    # Create a new column indicating significant arrival delays
    .withColumn(
        "IsDelayed",
        F.when(F.col("ArrDelay") > 15, 1).otherwise(0)
    )
)

display(flights_clean.limit(10))


Year,Month,UniqueCarrier,Origin,Dest,Distance,ArrDelay,DepDelay,IsDelayed
2007,1,WN,SMF,ONT,389,1.0,7.0,0
2007,1,WN,SMF,PDX,479,8.0,13.0,0
2007,1,WN,SMF,PDX,479,34.0,36.0,1
2007,1,WN,SMF,PDX,479,26.0,30.0,1
2007,1,WN,SMF,PDX,479,-3.0,1.0,0
2007,1,WN,SMF,PDX,479,3.0,10.0,0
2007,1,WN,SMF,PHX,647,47.0,56.0,1
2007,1,WN,SMF,PHX,647,-2.0,9.0,0
2007,1,WN,SMF,PHX,647,44.0,47.0,1
2007,1,WN,SMF,PHX,647,-7.0,3.0,0


## 3. DataFrame Aggregations

PySpark `groupBy()` and aggregation functions are used to analyze flight performance by airline and month.

The airline-level analysis calculates flight counts, average arrival and departure delays, delay percentages, and average flight distances.

The monthly analysis calculates total flights, average arrival delays, and delay rates to identify seasonal patterns.

### Key Findings

- December had the highest monthly delay rate at **25.32%**.
- September had the lowest monthly delay rate at **15.19%**.
- June had the highest average arrival delay at **11.28 minutes**.
- Among the airlines analyzed, EV had the highest delay rate at **25.46%**, while HA had the lowest at **6.12%**.

These results demonstrate how distributed aggregation can summarize millions of records into interpretable performance metrics.

In [0]:

# Aggregate flight performance by airline
airline_stats = (
    flights_clean
    .groupBy("UniqueCarrier")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrivalDelay"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepartureDelay"),
        F.round(F.avg("IsDelayed") * 100, 2).alias("DelayRatePct"),
        F.round(F.avg("Distance"), 2).alias("AvgDistance")
    )
    .orderBy(F.desc("DelayRatePct"))
)

display(airline_stats)


UniqueCarrier,TotalFlights,AvgArrivalDelay,AvgDepartureDelay,DelayRatePct,AvgDistance
EV,1645093,10.74,13.51,25.46,451.17
B6,799117,9.67,11.22,24.0,1195.82
YV,822290,11.44,12.89,23.18,398.8
AS,1539027,8.57,9.6,23.01,838.3
FL,1249316,9.48,10.25,22.54,666.9
UA,5695168,9.27,10.51,22.31,1015.67
MQ,3789853,8.56,9.18,22.23,366.4
XE,2290461,9.04,8.41,21.92,538.13
OH,1414120,7.48,9.28,21.52,469.55
HP,1385572,7.62,9.1,21.44,943.51


In [0]:

# Aggregate flight performance by month
monthly_stats = (
    flights_clean
    .groupBy("Month")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrivalDelay"),
        F.round(F.avg("IsDelayed") * 100, 2).alias("DelayRatePct")
    )
    .orderBy("Month")
)

display(monthly_stats)


Month,TotalFlights,AvgArrivalDelay,DelayRatePct
1,5245648,7.57,21.27
2,4846378,7.83,21.56
3,5442199,7.15,20.34
4,5277090,5.24,17.88
5,5411121,5.73,18.13
6,5338923,11.28,23.97
7,5529206,10.45,22.92
8,5562497,8.5,21.15
9,5055751,3.08,15.19
10,5323521,4.43,16.95


## 4. Spark SQL Analysis

A temporary SQL view is created from the cleaned PySpark DataFrame to demonstrate querying distributed data using Spark SQL.

**SQL Query 1: Airport Delay Analysis**

Groups flights by departure airport, calculates delay statistics, and identifies the 10 airports with the highest delay rates among airports with at least 100,000 qualifying flights.

JFK had the highest delay rate in this comparison at **27.57%**, followed by PHL at **27.18%** and ORD at **26.98%**.

**SQL Query 2: Yearly Delay Trends**

Groups flights by year to calculate total flights, average arrival and departure delays, and delay percentages.

Among the years analyzed, 2007 had the highest delay rate at **23.84%**, while 2003 had the lowest at **15.77%**.

Both queries use Spark SQL aggregations, filtering, and ordering to produce analytical summaries.

In [0]:
flights_clean.createOrReplaceTempView("flights_clean_view")

In [0]:
airport_delays = spark.sql("""
    SELECT
        Origin,
        COUNT(*) AS TotalFlights,
        ROUND(AVG(ArrDelay), 2) AS AvgArrivalDelay,
        ROUND(AVG(IsDelayed) * 100, 2) AS DelayRatePct
    FROM flights_clean_view
    GROUP BY Origin
    HAVING COUNT(*) >= 100000
    ORDER BY DelayRatePct DESC
    LIMIT 10
""")

display(airport_delays)

Origin,TotalFlights,AvgArrivalDelay,DelayRatePct
JFK,802881,10.65,27.57
PHL,1095836,10.57,27.18
ORD,3303128,12.45,26.98
EWR,1337617,11.01,26.68
ATL,3389571,10.59,24.64
LGA,1086041,8.29,24.55
DTW,1452041,9.53,24.51
MIA,644328,9.59,22.79
MSP,1386975,7.42,22.57
IAD,830248,8.61,22.44


In [0]:

# SQL Query 2: Analyze flight delays by year
yearly_delays = spark.sql("""
    SELECT
        Year,
        COUNT(*) AS TotalFlights,
        ROUND(AVG(ArrDelay), 2) AS AvgArrivalDelay,
        ROUND(AVG(DepDelay), 2) AS AvgDepartureDelay,
        ROUND(AVG(IsDelayed) * 100, 2) AS DelayRatePct
    FROM flights_clean_view
    GROUP BY Year
    ORDER BY Year
""")

display(yearly_delays)


Year,TotalFlights,AvgArrivalDelay,AvgDepartureDelay,DelayRatePct
1999,5360018,8.25,9.28,20.55
2000,5481303,10.47,11.21,23.75
2001,5723673,5.53,8.12,18.41
2002,5197860,3.19,5.5,15.84
2003,6375689,3.6,5.22,15.77
2004,6987729,6.51,7.86,19.41
2005,6992838,7.18,8.65,20.01
2006,7003802,8.68,10.06,22.11
2007,7275288,10.19,11.36,23.84
2008,6855029,8.17,9.92,21.39


## 5. Physical Execution Plan and Query Optimization

Spark's `.explain(mode="formatted")` method is used to inspect the physical execution plan of the airport-delay query.

The execution plan demonstrates several optimization techniques:

- **Filter pushdown:** Spark's physical execution plan shows predicates for cancelled and diverted flights under `PushedFilters` at the CSV scan. Although these filters are pushed toward the data source, CSV files do not provide the same data-skipping advantages as Parquet.
- **Column pruning:** The airport-delay query reads only five required columns rather than all 29 source columns.
- **Partial aggregation:** Spark performs aggregation before and after redistributing data to reduce intermediate data volume.
- **Photon execution:** Databricks uses its optimized Photon execution engine.

The Databricks Query Details view reported a wall-clock duration of **20.68 seconds**, approximately **5.85 GB read**, and **63.25 million rows read** for the airport-delay query.

The source scan and shuffle operations were potential performance bottlenecks. Early filtering, column pruning, and partial aggregation helped reduce unnecessary processing and data movement.

In [0]:

# Examine the physical execution plan
print("=== AIRPORT DELAYS EXECUTION PLAN ===")

airport_delays.explain(mode="formatted")


=== AIRPORT DELAYS EXECUTION PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   PhotonResultStage (18)
   +- PhotonColumnarToRow (17)
      +- PhotonTopK (16)
         +- PhotonShuffleExchangeSource (15)
            +- PhotonShuffleMapStage (14)
               +- PhotonShuffleExchangeSink (13)
                  +- PhotonTopK (12)
                     +- PhotonFilter (11)
                        +- PhotonGroupingAgg (10)
                           +- PhotonShuffleExchangeSource (9)
                              +- PhotonShuffleMapStage (8)
                                 +- PhotonShuffleExchangeSink (7)
                                    +- PhotonGroupingAgg (6)
                                       +- PhotonProject (5)
                                          +- PhotonProject (4)
                                             +- PhotonFilter (3)
                                                +- PhotonRowToColumnar (2)
                                       

### Performance Analysis

Spark's physical execution plan demonstrates filter pushdown, column pruning, and partial aggregation. The filters excluding cancelled and diverted flights are pushed toward the CSV scan, along with null checks. Column pruning reduces the airport-delay query's required input columns from 29 to 5. The execution plan also shows Databricks Photon operators and adaptive query execution, indicating that Spark uses optimized physical operators for the query.

The Databricks Query Details view reports an execution time of 20.68 seconds, approximately 5.85 GB of data read, and 63.25 million input rows. The large CSV scan and shuffle required to group flights by departure airport are potential performance bottlenecks. To reduce unnecessary processing, the pipeline applies filters and column selection early, performs partial aggregation before shuffling, and avoids unnecessary joins. The cleaned dataset is also written to Parquet partitioned by year, enabling partition pruning for year-specific queries.

Lazy evaluation was demonstrated by defining a filtering transformation in 0.0005 seconds, while the subsequent `count()` action took 18.73 seconds. A caching experiment showed that the second uncached aggregation took 0.56 seconds compared with 0.65 seconds for the cached repeated run. Therefore, caching did not improve performance in this particular experiment. This illustrates why caching should be benchmarked rather than assumed to accelerate every workload.

## 6. Actions vs. Transformations: Lazy Evaluation

Spark transformations are evaluated lazily, while actions trigger execution.

To demonstrate this behavior, the pipeline defines a transformation that filters flights with arrival delays greater than 15 minutes. The transformation is then evaluated using the `count()` action.

**Measured results:**

| Operation | Execution time |
|---|---:|
| Define filtering transformation | 0.0005 seconds |
| Execute `count()` action | 18.73 seconds |

The action returned **12,790,687 delayed flights**.

The substantial difference in execution time demonstrates that defining a transformation builds a logical execution plan, whereas an action triggers the actual distributed computation.

In [0]:

import time

# TRANSFORMATION: Spark builds a logical plan
start = time.perf_counter()

delayed_flights = flights_clean.filter(
    F.col("ArrDelay") > 15
)

transformation_time = time.perf_counter() - start

print("Transformation defined.")
print(f"Time: {transformation_time:.4f} seconds")
print("No full dataset computation has been triggered yet.")

# ACTION: Spark executes the computation
start = time.perf_counter()

delayed_count = delayed_flights.count()

action_time = time.perf_counter() - start

print(f"\nDelayed flights: {delayed_count:,}")
print(f"Action execution time: {action_time:.2f} seconds")


Transformation defined.
Time: 0.0009 seconds
No full dataset computation has been triggered yet.

Delayed flights: 12,790,687
Action execution time: 18.38 seconds


## 7. Parquet Output, Partition Pruning, and Caching

### Parquet Output

The cleaned dataset is written to a Unity Catalog volume in Parquet format:

`/Volumes/workspace/default/ids706_week12/cleaned_flights`

The output contains **63,253,229 cleaned flight records** and is partitioned by year (1999–2008).

Parquet provides columnar storage, while year-based partitioning allows queries restricted to specific years to avoid scanning unrelated partitions.

### Partition Pruning

A query filtering for 2007 demonstrates partition pruning. The physical execution plan includes:

`PartitionFilters: [isnotnull(Year), (Year = 2007)]`

The query returned **7,275,288 flights** for 2007.

This confirms that Spark recognizes the year partition and can restrict the scan to relevant data.

### Caching Experiment

A repeated airline aggregation was executed with and without explicit caching.

| Operation | Time |
|---|---:|
| First uncached run | 0.97 s |
| Second uncached run | 0.56 s |
| Cache materialization | 0.84 s |
| Cached repeated run | 0.65 s |

Caching did not improve performance in this experiment. The aggregation result was relatively small, and the differences may reflect runtime variability or existing execution optimizations.

The experiment demonstrates that caching has overhead and should be evaluated for the specific workload rather than assumed to improve performance universally.

In [0]:

# Check available catalogs and schemas
spark.sql("SHOW CATALOGS").show(truncate=False)

print("Current catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current schema:", spark.sql("SELECT current_schema()").first()[0])


+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+

Current catalog: workspace
Current schema: default


In [0]:

# Create a managed volume for the assignment
spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.default.ids706_week12
""")

print("Volume created successfully!")


Volume created successfully!


In [0]:

# Save cleaned flight data as Parquet, partitioned by year
output_path = "/Volumes/workspace/default/ids706_week12/cleaned_flights"

(
    flights_clean
    .write
    .mode("overwrite")
    .partitionBy("Year")
    .parquet(output_path)
)

print("Successfully saved cleaned flights!")
print("Output path:", output_path)


Successfully saved cleaned flights!
Output path: /Volumes/workspace/default/ids706_week12/cleaned_flights


In [0]:

# Read the saved Parquet dataset
saved_flights = spark.read.parquet(output_path)

print(f"Saved flight count: {saved_flights.count():,}")

# Show partition folders
display(dbutils.fs.ls(output_path))


Saved flight count: 63,253,229


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=1999/,Year=1999/,0,1795464838290
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=2000/,Year=2000/,0,1795464838290
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=2001/,Year=2001/,0,1795464838290
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=2002/,Year=2002/,0,1795464838290
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=2003/,Year=2003/,0,1795464838290
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=2004/,Year=2004/,0,1795464838290
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=2005/,Year=2005/,0,1795464838290
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=2006/,Year=2006/,0,1795464838290
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=2007/,Year=2007/,0,1795464838290
dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights/Year=2008/,Year=2008/,0,1795464838290


In [0]:
# Filter to a single year
flights_2007 = saved_flights.filter(
    F.col("Year") == 2007
)

# Examine the execution plan
print("=== PARTITION PRUNING EXECUTION PLAN ===")
flights_2007.explain(mode="formatted")

# Trigger execution
print("Flights in 2007:", flights_2007.count())


=== PARTITION PRUNING EXECUTION PLAN ===
== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [9]: [Month#13722, UniqueCarrier#13723, Origin#13724, Dest#13725, Distance#13726, ArrDelay#13727, DepDelay#13728, IsDelayed#13729, Year#13730]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/ids706_week12/cleaned_flights]
PartitionFilters: [isnotnull(Year#13730), (Year#13730 = 2007)]
ReadSchema: struct<Month:int,UniqueCarrier:string,Origin:string,Dest:string,Distance:int,ArrDelay:double,DepDelay:double,IsDelayed:int>

(2) PhotonColumnarToRow
Input [9]: [Month#13722, UniqueCarrier#13723, Origin#13724, Dest#13725, Distance#13726, ArrDelay#13727, DepDelay#13728, IsDelayed#13729, Year#13730]

(3) PhotonResultStage
Input [9]: [Month#13722, UniqueCarrier#13723, Origin#13724, Dest#13725, Distance#13726, ArrDelay#13727, DepDelay#13728, IsDelayed#13729, Year#13730]


== Photon Explanation ==
The query is fully s

### Caching Experiment

The following experiment compares repeated execution of an airline-level aggregation with and without explicit Spark caching. Execution times are measured using `time.perf_counter()`. The purpose is to evaluate whether caching reduces execution time for this particular workload.

In [0]:

import time

# Use a reusable aggregation without an explicit sort
cache_test = (
    saved_flights
    .groupBy("UniqueCarrier")
    .agg(F.avg("ArrDelay").alias("AvgDelay"))
)

# Compare repeated actions without caching
start = time.perf_counter()
cache_test.collect()
uncached_first = time.perf_counter() - start

start = time.perf_counter()
cache_test.collect()
uncached_second = time.perf_counter() - start

# Cache the aggregated results
cached_test = cache_test.cache()

start = time.perf_counter()
cached_test.collect()  # Materializes the cache
cache_build = time.perf_counter() - start

start = time.perf_counter()
cached_test.collect()
cached_second = time.perf_counter() - start

print(f"Uncached first run: {uncached_first:.2f}s")
print(f"Uncached second run: {uncached_second:.2f}s")
print(f"Cache materialization: {cache_build:.2f}s")
print(f"Cached repeated run: {cached_second:.2f}s")

cached_test.unpersist()


Uncached first run: 0.76s
Uncached second run: 0.46s
Cache materialization: 0.57s
Cached repeated run: 0.48s


## 8. Optional Machine Learning: Flight Delay Prediction

A logistic regression classifier was implemented using Spark MLlib to predict whether a flight would arrive more than 15 minutes late.

The model used month, flight distance, airline, and departure airport as features. Categorical variables were encoded using `StringIndexer` and `OneHotEncoder`, and features were combined using `VectorAssembler`.

A reproducible sample of **189,189 flights** was split into 80% training and 20% testing data.

### Model Results

| Metric | Result |
|---|---:|
| Classification accuracy | 79.79% |
| ROC AUC | 0.5657 |
| Majority-class baseline accuracy | 79.79% |
| Accuracy improvement | 0.00 percentage points |

Although the model achieved 79.79% accuracy, it did not outperform a baseline that always predicts the majority class.

The ROC AUC of 0.5657 indicates limited discrimination between delayed and non-delayed flights.

These findings demonstrate the importance of evaluating machine learning models using appropriate metrics, especially when the target classes are imbalanced.

Future improvements could include scheduled departure times, route characteristics, and historical airport delay patterns.

In [0]:

# Step 29: Prepare data for machine learning

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.sql import functions as F

# Select relevant features and target variable
ml_data = (
    saved_flights
    .select(
        "Month",
        "Distance",
        "UniqueCarrier",
        "Origin",
        "IsDelayed"
    )
    .filter(F.col("Distance") > 0)
    .dropna()
    .sample(withReplacement=False, fraction=0.003, seed=42)
)

print("ML dataset size:", ml_data.count())
display(ml_data.limit(5))


ML dataset size: 189189


Month,Distance,UniqueCarrier,Origin,IsDelayed
1,257,US,ROC,1
1,2378,US,SEA,0
1,920,US,TPA,0
1,920,US,TPA,0
1,328,WN,ABQ,1


In [0]:

# Step 30: Train a logistic regression model

# Convert categorical variables into numeric features
carrier_indexer = StringIndexer(
    inputCol="UniqueCarrier",
    outputCol="CarrierIndex",
    handleInvalid="keep"
)

origin_indexer = StringIndexer(
    inputCol="Origin",
    outputCol="OriginIndex",
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCols=["CarrierIndex", "OriginIndex"],
    outputCols=["CarrierVector", "OriginVector"]
)

# Combine features into a single vector
assembler = VectorAssembler(
    inputCols=[
        "Month",
        "Distance",
        "CarrierVector",
        "OriginVector"
    ],
    outputCol="features"
)

# Logistic regression classifier
lr = LogisticRegression(
    featuresCol="features",
    labelCol="IsDelayed",
    maxIter=20,
    regParam=0.1
)

# Build the ML pipeline
ml_pipeline = Pipeline(
    stages=[
        carrier_indexer,
        origin_indexer,
        encoder,
        assembler,
        lr
    ]
)

# Split data into training and testing sets
train_data, test_data = ml_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Train the model
ml_model = ml_pipeline.fit(train_data)

print("Logistic regression model trained successfully!")


Logistic regression model trained successfully!


In [0]:

# Step 31: Evaluate model performance

predictions = ml_model.transform(test_data)

# Calculate accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="IsDelayed",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(predictions)

# Calculate area under ROC curve
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="IsDelayed",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(predictions)

print(f"Model Accuracy: {accuracy:.4f}")
print(f"ROC AUC: {auc:.4f}")

# Show example predictions
display(
    predictions.select(
        "Month",
        "Distance",
        "UniqueCarrier",
        "Origin",
        "IsDelayed",
        "prediction",
        "probability"
    ).limit(10)
)


Model Accuracy: 0.7979
ROC AUC: 0.5657


Month,Distance,UniqueCarrier,Origin,IsDelayed,prediction,probability
1,72,UA,DEN,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8018103205698757"",""0.19818967943012433""]}"
1,75,AA,RIC,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7893741316895437"",""0.21062586831045627""]}"
1,75,UA,PWM,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7687090292261809"",""0.2312909707738191""]}"
1,83,DL,SDF,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8192690892793136"",""0.18073091072068637""]}"
1,88,US,CLT,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7845235870620871"",""0.21547641293791286""]}"
1,89,UA,SBA,1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.805527559074937"",""0.194472440925063""]}"
1,90,US,PHL,1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7518901228737334"",""0.24810987712626664""]}"
1,95,NW,CLE,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.824324524027307"",""0.17567547597269295""]}"
1,105,DL,MLU,0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8521410289815404"",""0.14785897101845957""]}"
1,109,US,PIT,1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7803397629825132"",""0.21966023701748683""]}"


In [0]:

# Step 32: Compare model with majority-class baseline

from pyspark.sql import functions as F

# Find the majority class in training data
train_counts = (
    train_data
    .groupBy("IsDelayed")
    .count()
    .orderBy(F.desc("count"))
    .collect()
)

majority_class = train_counts[0]["IsDelayed"]

# Baseline accuracy on test data
baseline_accuracy = (
    test_data
    .filter(F.col("IsDelayed") == majority_class)
    .count()
    / test_data.count()
)

print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print(f"Model Accuracy: {accuracy:.4f}")
print(f"Model ROC AUC: {auc:.4f}")
print(
    f"Accuracy improvement: "
    f"{(accuracy - baseline_accuracy) * 100:.2f} percentage points"
)


Baseline Accuracy: 0.7979
Model Accuracy: 0.7979
Model ROC AUC: 0.5657
Accuracy improvement: 0.00 percentage points
